# P4 Agent 4 · NCS Mapping Export

| 항목 | 명세 |
|---|---|
| 목적 | Build six canonical Parquet and UTF-8-SIG inspection CSV pairs. |
| 담당 Agent | `P4-A4-NCS` |
| Stage ID | `A4-04-EXPORT` |
| 입력 | `ncs_mapping/data/processed/observed-dev/NCS_MAPPING_OBSERVED_20260806_01/posting_ncs_candidates.parquet` |
| 처리 | 6개 canonical Parquet 및 inspection CSV semantic export |
| 출력 | six CSV/Parquet pairs and handoff 및 4개 종료 artifact |
| 선행 Gate | `NCS_MAPPING_DEV_READY` |
| 후속 활용 | Agent 2 preprocessing integration |

> Development-only orchestration. Empirical analysis and production promotion are disabled.

In [1]:
RUN_MODE = "observed-dev"
AGENT_ID = "P4-A4-NCS"
STAGE_ID = "A4-04-EXPORT"
CONTRACT_VERSION = "2.1.2"
SCHEMA_VERSION = "ncs-export-v1"
DATA_VERSION = "observed-dev-20260806.1"
CRAWL_RELEASE_ID = "CRAWL_20260806_03"
AS_OF_DATE = "2026-08-06"
INPUT_MANIFEST_PATH = "ncs_mapping/data/processed/observed-dev/NCS_MAPPING_OBSERVED_20260806_01/posting_ncs_candidates.parquet"
OUTPUT_ROOT = "ncs_mapping/data/runs/observed-dev/NCS_MAPPING_OBSERVED_20260806_01/A4-04-EXPORT"
RANDOM_SEED = 20260806
FAIL_ON_GATE = True
EMPIRICAL_ANALYSIS_ALLOWED = False
DATA_PROVENANCE = "OBSERVED_DEVELOPMENT_ONLY"
PROMOTION_ALLOWED = False
DUTY_INPUT_PATH = ""
GOLD_INPUT_PATH = ""
CONTROL_SCHEMA_DIR = ""

# Injected into the executed copy by crawl.control.notebook_bundle
RUN_MODE = 'observed-dev'
FAIL_ON_GATE = True
EMPIRICAL_ANALYSIS_ALLOWED = False

In [2]:
from pathlib import Path
import os
import sys
import pandas as pd

NCS_ROOT = Path.cwd().resolve()
if NCS_ROOT.name != 'ncs_mapping':
    raise RuntimeError('run this notebook with cwd=ncs_mapping')
sys.path.insert(0, str(NCS_ROOT / 'src'))
assert RUN_MODE == 'observed-dev'
assert AGENT_ID == 'P4-A4-NCS' and STAGE_ID.startswith('A4-')
assert RANDOM_SEED == 20260806 and FAIL_ON_GATE is True
assert DATA_PROVENANCE == 'OBSERVED_DEVELOPMENT_ONLY'
assert EMPIRICAL_ANALYSIS_ALLOWED is False and PROMOTION_ALLOWED is False
resolved_duty_input = DUTY_INPUT_PATH or os.environ.get('P4_A2_DUTY_HANDOFF', '')
resolved_gold_input = GOLD_INPUT_PATH or os.environ.get('P4_NCS_GOLD_INPUT', '')
resolved_schema_dir = CONTROL_SCHEMA_DIR or os.environ.get('P4_CONTROL_SCHEMA_DIR', '')

In [3]:
from p4_ncs.quality.stage_artifacts import sha256_file

processed_root = NCS_ROOT / 'data/processed/observed-dev/NCS_MAPPING_OBSERVED_20260806_01'
candidate_path = processed_root / 'posting_ncs_candidates.parquet'
match_path = processed_root / 'posting_ncs_matches.parquet'
input_audit = {'candidateRows': len(pd.read_parquet(candidate_path)), 'matchRows': len(pd.read_parquet(match_path)), 'candidateSha256': sha256_file(candidate_path), 'matchSha256': sha256_file(match_path)}
assert input_audit['matchRows'] == 28
input_audit

{'candidateRows': 128,
 'matchRows': 28,
 'candidateSha256': 'aebb2ff50f48eec06eb98b24ddc3a022ba694d6010830a2e5aae56d738ce8166',
 'matchSha256': 'ed3b4f360c370a8445fb7ac02c875171b24e3f9b707a2e3ad5ee82f5385be528'}

In [4]:
from p4_ncs.workflow.observed import run_stage

stage_manifest = run_stage('A4-04-EXPORT', root=NCS_ROOT, duty_input_path=resolved_duty_input or None, gold_input_path=resolved_gold_input or None, schema_dir=resolved_schema_dir or None)
stage_manifest

{'manifestVersion': 'stage-manifest-v1',
 'runId': 'NCS_MAPPING_OBSERVED_20260806_01',
 'runMode': 'observed-dev',
 'stageId': 'A4-04-EXPORT',
 'status': 'SUCCEEDED',
 'agentId': 'P4-A4-NCS',
 'branch': 'DETACHED_HEAD',
 'gitHead': '71edc867a8e869b78340e4fe253ced9312894077',
 'contractVersion': '2.1.2',
 'schemaVersion': 'ncs-export-v1',
 'dataVersion': 'observed-dev-20260806.1',
 'crawlReleaseId': 'CRAWL_20260806_03',
 'dataProvenance': 'OBSERVED_DEVELOPMENT_ONLY',
 'startedAt': '2026-08-06T09:16:57.321359Z',
 'completedAt': '2026-08-06T09:16:57.322550Z',
 'empiricalAnalysisAllowed': False,
 'promotionAllowed': False,
 'inputManifestSha256': 'a305d2354a7437860dc947f4e537be76c6233cd26fe658da650d3bb07437bc30',
 'parameterSha256': 'ee2a95b4cf722fbc15def7f7c955eba2248f681ec2a54ffef33e1869c3836e36',
 'rowCounts': {'ncs_units': 13442,
  'core_ai_it_codes': 120,
  'ncs_alias_dictionary': 10,
  'posting_ncs_candidates': 128,
  'posting_ncs_matches': 28,
  'ncs_mapping_summary': 1},
 'gateResu

In [5]:
stage_root = NCS_ROOT / 'data/runs' / RUN_MODE / 'NCS_MAPPING_OBSERVED_20260806_01' / stage_manifest['stageId']
expected_artifacts = {'stage_manifest.json', 'stage_metrics.json', 'stage_quality.csv', 'CHECKSUMS.sha256'}
actual_artifacts = {path.name for path in stage_root.iterdir() if path.is_file()}
assert actual_artifacts == expected_artifacts
termination_summary = {'stageId': stage_manifest['stageId'], 'status': stage_manifest['status'], 'rowCounts': stage_manifest['rowCounts'], 'artifacts': sorted(actual_artifacts)}
termination_summary

{'stageId': 'A4-04-EXPORT',
 'status': 'SUCCEEDED',
 'rowCounts': {'ncs_units': 13442,
  'core_ai_it_codes': 120,
  'ncs_alias_dictionary': 10,
  'posting_ncs_candidates': 128,
  'posting_ncs_matches': 28,
  'ncs_mapping_summary': 1},
 'artifacts': ['CHECKSUMS.sha256',
  'stage_manifest.json',
  'stage_metrics.json',
  'stage_quality.csv']}